# Notebook 02 — Feature Engineering
**Student 2 · Feature Engineer**

Builds a fully leak-free feature set from raw price/volume data, runs a formal
leakage test, produces descriptive statistics and a correlation heatmap, then
saves `data/processed/features.csv`.

**Prerequisites:** Run `01_data.ipynb` first.

In [ ]:
# ── Cell 1 · Setup & Imports ──────────────────────────────────────────────────
import sys, os
sys.path.insert(0, os.path.abspath('../src'))

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

DATA_DIR    = '../data'
RAW_DIR     = os.path.join(DATA_DIR, 'raw')
PROC_DIR    = os.path.join(DATA_DIR, 'processed')
RESULTS_DIR = '../results'
os.makedirs(RESULTS_DIR, exist_ok=True)

print('Setup complete.')

In [ ]:
# ── Cell 2 · Load Data ────────────────────────────────────────────────────────
returns_df = pd.read_csv(os.path.join(PROC_DIR, 'returns.csv'),
                         index_col='date', parse_dates=True)
prices_df  = pd.read_csv(os.path.join(RAW_DIR, 'prices.csv'),
                         index_col='date', parse_dates=True)
index_df   = pd.read_csv(os.path.join(RAW_DIR, 'index.csv'),
                         index_col='date', parse_dates=True)
universe_df = pd.read_csv(os.path.join(RAW_DIR, 'universe.csv'))

try:
    volumes_df = pd.read_csv(os.path.join(RAW_DIR, 'volumes.csv'),
                             index_col='date', parse_dates=True)
except FileNotFoundError:
    volumes_df = pd.DataFrame(index=prices_df.index, columns=prices_df.columns, dtype=float)

nifty_returns = index_df['Return'] if 'Return' in index_df.columns else \
                index_df['Close'].pct_change()

TICKERS = list(prices_df.columns)

print(f'prices_df  : {prices_df.shape}')
print(f'returns_df : {returns_df.shape}')
print(f'volumes_df : {volumes_df.shape}')
print(f'nifty_ret  : {nifty_returns.shape}')
print(f'Tickers    : {len(TICKERS)}')

In [ ]:
# ── Cell 3 · Import Feature Functions from src/features.py ───────────────────
from features import (
    compute_past_returns,
    compute_rolling_volatility,
    compute_momentum,
    compute_rsi,
    compute_volume_change,
    compute_dist_52wh,
    compute_rel_nifty,
    build_target,
    run_leakage_test,
    build_features_table,
)
print('Feature functions imported successfully from src/features.py')

In [ ]:
# ── Cell 4 · Compute Features for Each Ticker (Demo: single ticker) ───────────
demo_ticker = TICKERS[0]
prices_s  = prices_df[demo_ticker]
returns_s = returns_df[demo_ticker]
volume_s  = volumes_df[demo_ticker] if demo_ticker in volumes_df.columns else \
            pd.Series(dtype=float, index=prices_df.index)

demo_feats = pd.DataFrame(index=prices_df.index)

past_rets   = compute_past_returns(prices_s, windows=[1, 5, 21, 63])
roll_vols   = compute_rolling_volatility(returns_s, windows=[21, 63])
mom         = compute_momentum(prices_s)
rsi         = compute_rsi(prices_s)
volchg      = compute_volume_change(volume_s)
dist_52wh   = compute_dist_52wh(prices_s)
rel_nifty   = compute_rel_nifty(returns_s, nifty_returns)
target      = build_target(prices_s, horizon=21)

demo_feats = pd.concat([past_rets, roll_vols,
                        mom.rename('f_mom_126_21'),
                        rsi.rename('f_rsi_14'),
                        volchg.rename('f_volchg'),
                        dist_52wh.rename('f_dist_52wh'),
                        rel_nifty.rename('f_rel_nifty'),
                        target.rename('target_21d')], axis=1)

print(f'Demo feature table for {demo_ticker}: {demo_feats.shape}')
print(demo_feats.tail(5).round(4))

In [ ]:
# ── Cell 5 · Leakage Test ─────────────────────────────────────────────────────
TRUNCATION_DATES = [
    '2016-06-30',
    '2018-03-31',
    '2019-12-31',
    '2021-06-30',
    '2022-12-31',
    '2024-03-31',
]

print('Running leakage test …')
leakage_result = run_leakage_test(
    prices_df=prices_df,
    volume_df=volumes_df,
    nifty_returns=nifty_returns,
    truncation_dates=TRUNCATION_DATES
)

print(f"\nLeakage test PASSED: {leakage_result['passed']}")
if 'details' in leakage_result:
    for d, info in leakage_result['details'].items():
        status = '✅ PASS' if info.get('passed', False) else '❌ FAIL'
        print(f'  {d}  {status}  max_diff={info.get("max_diff", "N/A")}')

import json
with open(os.path.join(RESULTS_DIR, 'leakage_test.json'), 'w') as f:
    json.dump(leakage_result, f, indent=2, default=str)
print('\nLeakage test result saved → results/leakage_test.json')

In [ ]:
# ── Cell 6 · Build Full Features Table (All Tickers, Long Format) ─────────────
print('Building full features table … (this may take a minute)')
features_df = build_features_table(
    prices_df=prices_df,
    volume_df=volumes_df,
    nifty_returns=nifty_returns
)

print(f'Features table shape : {features_df.shape}')
print(f'Columns              : {list(features_df.columns)}')
print(f'Date range           : {features_df["date"].min().date()} → {features_df["date"].max().date()}')
print(f'Unique tickers       : {features_df["ticker"].nunique()}')
print(f'Target NaN rows      : {features_df["target_21d"].isna().sum()}')
print(features_df.head(3))

In [ ]:
# ── Cell 7 · Descriptive Statistics ──────────────────────────────────────────
feature_cols = [c for c in features_df.columns
                if c.startswith('f_') or c == 'target_21d']

desc = features_df[feature_cols].describe().T
desc['skew'] = features_df[feature_cols].skew()
desc['kurt'] = features_df[feature_cols].kurtosis()

print('=== Feature Descriptive Statistics ===')
print(desc.round(4).to_string())

# Histogram grid
n = len(feature_cols)
ncols = 4
nrows = (n + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(16, nrows * 3))
axes = axes.flatten()
for i, col in enumerate(feature_cols):
    axes[i].hist(features_df[col].dropna(), bins=60, edgecolor='none', alpha=0.7)
    axes[i].set_title(col, fontsize=9)
    axes[i].set_yticks([])
for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)
plt.suptitle('Feature Distributions', y=1.01, fontsize=13)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'feature_distributions.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Cell 8 · Correlation Heatmap ──────────────────────────────────────────────
corr = features_df[feature_cols].corr()

fig, ax = plt.subplots(figsize=(12, 10))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, vmin=-1, vmax=1, linewidths=0.5,
            annot_kws={'size': 8}, ax=ax)
ax.set_title('Feature Correlation Matrix (lower triangle)', fontsize=13)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'feature_correlations.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Saved → results/feature_correlations.png')

In [ ]:
# ── Cell 9 · Save features.csv ────────────────────────────────────────────────
out_path = os.path.join(PROC_DIR, 'features.csv')
features_df.to_csv(out_path, index=False)
print(f'Saved → {out_path}')
print(f'Shape  : {features_df.shape}')
print(f'Columns: {list(features_df.columns)}')
print('\n✅  Notebook 02 complete — features.csv saved.')